In [ ]:
import gradio as gr
import os
import torch
import numpy as np
import pandas as pd
import time
import json
import yaml
import sys
import subprocess
import re
import gc
from datetime import datetime
from PIL import Image, ImageEnhance, ImageDraw, ImageFilter

os.environ['no_proxy'] = 'localhost,127.0.0.1,::1'
os.environ['NO_PROXY'] = 'localhost,127.0.0.1,::1'

sys.path.append(os.path.abspath("."))

try:
    from src.core.registry import WATERMARKS, ATTACKS, MODELS
    from src.core.config_parser import ConfigManager
    from src.core.loader import setup_env
    setup_env()
except ImportError:
    print("Warning: src.core modules not found. Ensure you are in the correct directory.")


def list_files(subdir):
    path = os.path.join("configs", subdir)
    if not os.path.exists(path): return []
    return [f for f in os.listdir(path) if f.endswith('.yaml')]

def list_experiment_files():
    path = os.path.join("configs", "experiments")
    if not os.path.exists(path): return []

    return [f for f in os.listdir(path) if f.endswith('.yaml') and f.startswith('run_')]

GLOBAL_LISTS = {
    "wm": list_files("methods/watermarks"),
    "ds": list_files("datasets"),
    "exp": list_experiment_files(),
    "model": list_files("models")
}

class BenchmarkBackend:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self.watermark = None
        self.current_wm_name = None
        self.current_model_path = None
        self.current_secret = None
        self.CONFIG_ROOT = "configs/methods/watermarks"
        
        self.log_root = "outputs/history"
        self.single_log_dir = os.path.join(self.log_root, "single")
        self.batch_log_dir = os.path.join(self.log_root, "batch")
        os.makedirs(self.single_log_dir, exist_ok=True)
        os.makedirs(self.batch_log_dir, exist_ok=True)

    def _get_model_config(self, model_path):
        cfg = {
            "name": "StableDiffusion",
            "model_id": model_path,
            "device": self.device
        }
        if "v1-5" in model_path or "v2-1" in model_path: 
            cfg["dtype"] = "fp16"
        return cfg

    def _load_watermark_yaml(self, wm_name):
        if wm_name.endswith('.yaml'):
            yaml_path = os.path.join(self.CONFIG_ROOT, wm_name)
        else:
            yaml_path = os.path.join(self.CONFIG_ROOT, f"{wm_name}.yaml")
        if not os.path.exists(yaml_path):
            return {"name": wm_name.replace('.yaml', '')}
        return ConfigManager.load_yaml(yaml_path)

    def load_components(self, wm_name):
        status_msg = []
        try:
            wm_cfg = self._load_watermark_yaml(wm_name)
        except Exception as e:
            return False, f"Config Error: {e}"

        target_model_path = wm_cfg.get('model_path', "data/models/stable-diffusion-v1-5")
        
        if self.current_model_path != target_model_path:
            print(f">>> Auto-Loading Model: {target_model_path}...")
            try:
                if self.model is not None:
                    self.unload_components()
                
                model_cfg = self._get_model_config(target_model_path)
                self.model = MODELS.build(model_cfg)
                
                self.current_model_path = target_model_path
                status_msg.append(f"Model: {os.path.basename(target_model_path)}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                return False, f"Model Load Error: {e}"

        if self.current_wm_name != wm_name or self.watermark is None:
            print(f">>> Loading Watermark: {wm_name}...")
            try:
                wm_cfg['model_path'] = target_model_path
                wm_cfg['global_config'] = {
                    "device": self.device,
                    "model_config": self._get_model_config(target_model_path)
                }
                self.watermark = WATERMARKS.build(wm_cfg)
                self.current_wm_name = wm_name
                status_msg.append(f"Algorithm: {wm_name}")
            except Exception as e:
                import traceback
                traceback.print_exc()
                self.watermark = None 
                return False, f"WM Init Error: {e}"
        
        return True, " | ".join(status_msg)

    def unload_components(self):
        print(">>> [System] Aggressively releasing GPU resources...")
        if self.model is not None:
            try:
                if hasattr(self.model, 'pipe') and self.model.pipe is not None:
                    if hasattr(self.model.pipe, 'to'): self.model.pipe.to("cpu") 
                    del self.model.pipe
                elif hasattr(self.model, 'to'): self.model.to("cpu")
            except Exception as e: print(f"Warning during model offload: {e}")
            del self.model
            self.model = None

        if self.watermark is not None:
            if hasattr(self.watermark, 'model'):
                try:
                    if hasattr(self.watermark.model, 'to'): self.watermark.model.to("cpu")
                    del self.watermark.model
                except: pass
            del self.watermark
            self.watermark = None

        self.current_model_path = None
        self.current_wm_name = None
        self.current_secret = None
        gc.collect()
        torch.cuda.empty_cache()
        gc.collect()
        return "System Reset & Memory Cleared"

    
    def generate_and_embed(self, prompt, seed, wm_name):
        success, msg = self.load_components(wm_name)
        if not success: 
            return None, None, None, None, msg, seed, gr.update(elem_classes="btn-gray")
        
        secret_len = 32
        if hasattr(self.watermark, 'config') and 'secret_config' in self.watermark.config:
             secret_len = self.watermark.config['secret_config'].get('length', 32)
        
        secret = torch.randint(0, 2, (secret_len,), device=self.device).float()
        self.current_secret = secret 
        
        try:
            print(f">>> Generating...")
            pipeline = getattr(self.model, 'pipe', None)
            current_seed = int(seed)
            
            wm_img_list = self.watermark.embed(
                pipeline=pipeline, prompt=[prompt], secret=secret,
                seed=[current_seed], original_image=None
            )
            if not wm_img_list or wm_img_list[0] is None: 
                return None, None, None, None, "Empty Result", current_seed, gr.update(elem_classes="btn-gray")
            
            clean_wm_img = wm_img_list[0]
            return clean_wm_img, clean_wm_img, clean_wm_img, clean_wm_img, f"Success: {msg}", current_seed, gr.update(elem_classes="btn-done", value="✅ 1. Generated")
        except Exception as e:
            import traceback
            traceback.print_exc()
            return None, None, None, None, f"Runtime Error: {e}", seed, gr.update(elem_classes="btn-gray")

    def apply_attacks(self, hidden_clean_image, active_flags, params):
        if hidden_clean_image is None: 
            return None, None, "No Image Generated", gr.update(elem_classes="btn-gray")
        
        img = hidden_clean_image.copy().convert("RGB")
        log_str = []
        try:
            if active_flags[4]: 
                val = float(params['crop']); attacker = ATTACKS.build({"name": "CropRescale", "scale": val}); img = attacker.apply(img); log_str.append(f"Crop({val})")
            if active_flags[5]: 
                val = float(params['drop']); attacker = ATTACKS.build({"name": "RandomDrop", "ratio": val}); img = attacker.apply(img); log_str.append(f"Drop({val})")
            if active_flags[0]: 
                val = float(params['bright'])
                try: attacker = ATTACKS.build({"name": "Brightness", "brightness": val}); img = attacker.apply(img)
                except: img = ImageEnhance.Brightness(img).enhance(val)
                log_str.append(f"Bright({val})")
            if active_flags[2]: 
                val = float(params['blur']); attacker = ATTACKS.build({"name": "GaussianBlur", "radius": val}); img = attacker.apply(img); log_str.append(f"Blur({val})")
            if active_flags[3]: 
                val = float(params['noise']); attacker = ATTACKS.build({"name": "GaussianNoise", "std": val}); img = attacker.apply(img); log_str.append(f"Noise({val})")
            
            if active_flags[6]: 
                try:
                    val = int(params['vae_qual']); model_name = params.get('vae_model', 'bmshj2018-factorized')
                    attacker = ATTACKS.build({"name": "VAECompression", "quality": val, "model_name": model_name})
                    img = attacker.apply(img); log_str.append(f"VAE({model_name}, q={val})")
                except Exception as e: print(f"VAE Error: {e}")
            
            if active_flags[7]: 
                try:
                    val = float(params['diff_str']); attacker = ATTACKS.build({"name": "DiffusionRegeneration", "strength": val, "model_id": "stabilityai/stable-diffusion-2-1-base"})
                    img = attacker.apply(img); log_str.append(f"DiffRegen({val})")
                except Exception as e: print(f"DiffRegen Error: {e}")
            if active_flags[1]: 
                val = int(params['jpeg']); attacker = ATTACKS.build({"name": "JPEG", "quality": val}); img = attacker.apply(img); log_str.append(f"JPEG({val})")

            status_text = " + ".join(log_str) if log_str else "Clean"
            return img, img, status_text, gr.update(elem_classes="btn-done", value="✅ 2. Applied")
        except Exception as e:
            return hidden_clean_image, hidden_clean_image, f"Attack Error: {e}", gr.update(elem_classes="btn-gray")

    def extract_and_verify(self, hidden_target_image):
        if hidden_target_image is None: return "No Image in State", "Error", gr.update(elem_classes="btn-gray")
        if self.watermark is None and self.current_wm_name is not None: self.load_components(self.current_wm_name)
        if self.watermark is None: return "WM Not Loaded", "Error", gr.update(elem_classes="btn-gray")
        try:
            print(">>> Extracting...")
            raw_results = self.watermark.extract([hidden_target_image], secret=self.current_secret)
            final_metrics = self.watermark.compute_aggregate_metrics(raw_results)
            info_lines = []
            is_detected = False
            for k, v in final_metrics.items():
                val_str = f"{v:.4f}" if isinstance(v, float) else str(v)
                info_lines.append(f"{k}: {val_str}")
                if "TPR" in k or "Acc" in k:
                    if isinstance(v, (int, float)) and v >= 0.99: is_detected = True
            status = "✅ DETECTED" if is_detected else "❌ FAILED"
            return "\n".join(info_lines), status, gr.update(elem_classes="btn-done", value="✅ 3. Verified")
        except Exception as e:
            import traceback
            traceback.print_exc()
            return f"Error: {e}", "Crash", gr.update(elem_classes="btn-gray")

    
    def load_experiment_template_data(self, config_name):
        """
        [Internal]  UI 
        """
        def_vals = {
            'wm': "ZoDiac.yaml", 'ds': "hf_demo.yaml", 'seed': 42, 'bs': 2, 'max': 5, 'model': "sd_v1_5.yaml",
            'states': [False]*8, 'params': [1.2, 80, 1.0, 0.05, 0.9, 0.3, 3, 0.1]
        }
        
        if not config_name: return def_vals
        path = os.path.join("configs/experiments", config_name)
        if not os.path.exists(path): return def_vals
        
        try:
            with open(path, 'r') as f: exp_cfg = yaml.safe_load(f)
            defaults = exp_cfg.get('defaults', {})
            
            wm_path = defaults.get('method') or defaults.get('watermark_config') or 'ZoDiac.yaml'
            ds_path = defaults.get('dataset') or defaults.get('dataset_config') or 'hf_demo.yaml'
            model_path = defaults.get('model') or defaults.get('model_config') or 'sd_v1_5.yaml'
            
            wm_val = os.path.basename(wm_path)
            ds_val = os.path.basename(ds_path)
            model_val = os.path.basename(model_path)
            
            seed = exp_cfg.get('seed', 42)
            bs = exp_cfg.get('batch_size', 2)
            max_samples = exp_cfg.get('max_samples', 5)

            attacks = exp_cfg.get('attacks', [])
            p_vals = [1.2, 80, 1.0, 0.05, 0.9, 0.3, 3, 0.1]
            new_states = [False] * 8
            
            for atk in attacks:
                name = atk.get('name', '')
                if name == "Brightness": new_states[0] = True; p_vals[0] = float(atk.get('brightness', 1.2))
                elif name == "JPEG": new_states[1] = True; p_vals[1] = int(atk.get('quality', 80))
                elif name == "GaussianBlur": new_states[2] = True; p_vals[2] = float(atk.get('radius', 1.0))
                elif name == "GaussianNoise": new_states[3] = True; p_vals[3] = float(atk.get('std', 0.05))
                elif name == "CropRescale": new_states[4] = True; p_vals[4] = float(atk.get('scale', 0.9))
                elif name == "RandomDrop": new_states[5] = True; p_vals[5] = float(atk.get('ratio', 0.3))
                elif name == "VAECompression": new_states[6] = True; p_vals[6] = int(atk.get('quality', 3))
                elif name == "DiffusionRegeneration": new_states[7] = True; p_vals[7] = float(atk.get('strength', 0.1))
            
            return {
                'wm': wm_val, 'ds': ds_val, 'seed': seed, 'bs': bs, 'max': max_samples, 'model': model_val,
                'states': new_states, 'params': p_vals
            }
        except Exception as e:
            print(f"Error parsing template: {e}")
            return def_vals

    def wrap_ui_updates(self, data, update_template_dd=None):

        updates = []
        
        
        if update_template_dd:
            updates.append(gr.update(value=update_template_dd, choices=GLOBAL_LISTS['exp'], interactive=True))
        
        
        updates.append(gr.update(value=data['wm'], choices=GLOBAL_LISTS['wm'], interactive=True))
        
        
        updates.append(gr.update(value=data['ds'], choices=GLOBAL_LISTS['ds'], interactive=True))
        
        
        updates.append(gr.update(value=data['model'], choices=GLOBAL_LISTS['model'], interactive=True))
        
        
        updates.append(gr.update(value=data['seed']))
        updates.append(gr.update(value=data['bs']))
        updates.append(gr.update(value=data['max']))
        
        
        for s in data['states']:
            updates.append(gr.update(value=s))
            
        
        for p in data['params']:
            updates.append(gr.update(value=p))
            
        return updates

    def load_experiment_template_ui(self, config_name):
        """UI Event: Template Dropdown Change"""
        data = self.load_experiment_template_data(config_name)

        return self.wrap_ui_updates(data, update_template_dd=None)

    def find_template_by_watermark(self, wm_filename):
        """UI Event: Watermark Dropdown Change"""
        if not wm_filename: return [gr.update()] * 23 

        exp_dir = "configs/experiments"
        found_exp_file = None

        if os.path.exists(exp_dir):
            for f in os.listdir(exp_dir):
                if f.endswith('.yaml'):
                    try:
                        with open(os.path.join(exp_dir, f), 'r') as file:
                            cfg = yaml.safe_load(file)
                            defaults = cfg.get('defaults', {})
                            wm_path = defaults.get('method') or defaults.get('watermark_config')
                            if wm_path and os.path.basename(wm_path) == wm_filename:
                                found_exp_file = f
                                break
                    except: continue
        
        if found_exp_file:
            print(f">>> Auto-matching template: {found_exp_file}")
            data = self.load_experiment_template_data(found_exp_file)
  
            return self.wrap_ui_updates(data, update_template_dd=found_exp_file)
        else:
            return [gr.update()] * 23

    
    def run_batch_benchmark(self, wm_config_file, ds_config_file, model_config_file, batch_seed, batch_size, batch_max,
                            active_flags, params):
        self.unload_components()
        yield log_format("System", "Cleared GPU memory from Single Mode."), ""

        attacks_list = []
        attack_log_parts = [] 
        if active_flags[4]: attacks_list.append({"name": "CropRescale", "scale": float(params['crop'])}); attack_log_parts.append(f"Crop({params['crop']})")
        if active_flags[5]: attacks_list.append({"name": "RandomDrop", "ratio": float(params['drop'])}); attack_log_parts.append(f"Drop({params['drop']})")
        if active_flags[0]: attacks_list.append({"name": "Brightness", "brightness": float(params['bright'])}); attack_log_parts.append(f"Bright({params['bright']})")
        if active_flags[2]: attacks_list.append({"name": "GaussianBlur", "radius": float(params['blur'])}); attack_log_parts.append(f"Blur({params['blur']})")
        if active_flags[3]: attacks_list.append({"name": "GaussianNoise", "std": float(params['noise'])}); attack_log_parts.append(f"Noise({params['noise']})")
        
        if active_flags[6]: 
            v_qual = int(params['vae'])
            v_model = params.get('vae_model', 'bmshj2018-factorized')
            attacks_list.append({"name": "VAECompression", "quality": v_qual, "model_name": v_model})
            attack_log_parts.append(f"VAE({v_model}, q={v_qual})")
            
        if active_flags[7]: attacks_list.append({"name": "DiffusionRegeneration", "strength": float(params['diff']), "model_id": "stabilityai/stable-diffusion-2-1-base"}); attack_log_parts.append(f"Diff({params['diff']})")
        if active_flags[1]: attacks_list.append({"name": "JPEG", "quality": int(params['jpeg'])}); attack_log_parts.append(f"JPEG({params['jpeg']})")

        atk_chain_str = " + ".join(attack_log_parts) if attack_log_parts else "Clean"
        timestamp = int(time.time())
        wm_name_clean = os.path.basename(wm_config_file).replace('.yaml', '')
        target_model_config = f"configs/models/{model_config_file}"

        exp_config = {
            "experiment_name": f"{wm_name_clean}_Batch_{timestamp}",
            "output_dir": f"outputs/{wm_name_clean}_{timestamp}",
            "seed": int(batch_seed),
            "device": self.device,
            "batch_size": int(batch_size),
            "max_samples": int(batch_max),
            "defaults": {
                "model": target_model_config,
                "model_config": target_model_config, 
                "method": f"configs/methods/watermarks/{wm_config_file}",
                "watermark_config": f"configs/methods/watermarks/{wm_config_file}",
                "dataset": f"configs/datasets/{ds_config_file}",
                "dataset_config": f"configs/datasets/{ds_config_file}"
            },
            "attacks": attacks_list
        }
        
        os.makedirs("configs/temp", exist_ok=True)
        temp_yaml_path = os.path.abspath(f"configs/temp/temp_batch_{timestamp}.yaml")
        with open(temp_yaml_path, 'w') as f: yaml.dump(exp_config, f)
        
        cmd = [sys.executable, "-u", "main.py", "--config", temp_yaml_path]
        
        full_log = ""
        def log(text):
            nonlocal full_log
            full_log += text
            return full_log

        yield log(f"🚀 Initializing Batch Process...\n"), ""
        yield log(f"📂 Config: {temp_yaml_path}\n"), ""
        yield log(f"dataset: {ds_config_file}\n"), "" 
        yield log(f"⚔️ Attack Chain: {atk_chain_str}\n"), ""
        yield log(f"{'-'*40}\n"), ""
        
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=os.getcwd())
            for line in process.stdout: yield log(line), ""
            process.wait()
            if process.returncode == 0:
                metrics_found = []
                patterns = {
                    "avg_bit_acc": r"avg_bit_acc[:\s]+([\d\.]+)",
                    "TPR@1e-2": r"TPR@1e-2[:\s]+([\d\.]+)",
                    "TPR@1e-6": r"TPR@1e-6[:\s]+([\d\.]+)"
                }
                for name, pat in patterns.items():
                    match = re.search(pat, full_log)
                    if match: metrics_found.append(f"{name}: {match.group(1)}")
                metrics_str = ",\n".join(metrics_found) if metrics_found else "See Log"
                yield full_log, metrics_str
                self.save_log_entry(wm_config_file, ds_config_file, batch_seed, atk_chain_str, metrics_str, mode="batch")
            else:
                yield log(f"\n❌ Error {process.returncode}\n"), "Error"
        except Exception as e:
            yield log(f"\n❌ Exception: {str(e)}\n"), "Error"

    
    def get_log_path(self, wm_name, mode="single"):
        clean_name = wm_name.replace('.yaml', '') if wm_name else "general"
        base_dir = self.single_log_dir if mode == "single" else self.batch_log_dir
        return os.path.join(base_dir, f"{clean_name}_history.csv")

    def load_logs(self, wm_name, mode_radio="Single"):
        mode_key = "single" if mode_radio == "Single" else "batch"
        path = self.get_log_path(wm_name, mode_key)
        cols = ["Method", "Dataset Config", "Seed", "Attack Chain", "Metrics"]
        if os.path.exists(path):
            try: 
                df = pd.read_csv(path)
                if "Dataset" in df.columns: df = df.rename(columns={"Dataset": "Dataset Config"})
                valid_cols = [c for c in cols if c in df.columns]
                return df[valid_cols]
            except: pass
        return pd.DataFrame(columns=cols)

    def save_log_entry(self, wm_name, dataset_label, seed, atk_str, metrics_str, mode="single"):
        path = self.get_log_path(wm_name, mode)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        new_row = pd.DataFrame([[wm_name, dataset_label, seed, atk_str, metrics_str]], 
                               columns=["Method", "Dataset Config", "Seed", "Attack Chain", "Metrics"])
        if os.path.exists(path):
            try:
                df = pd.read_csv(path)
                if "Dataset" in df.columns: df = df.rename(columns={"Dataset": "Dataset Config"})
                if "Count" in df.columns: df = df.drop(columns=["Count"])
                df = pd.concat([df, new_row], ignore_index=True)
            except: df = new_row
        else: df = new_row
        df.to_csv(path, index=False)
        return df

    def overwrite_logs(self, wm_name, mode_radio, new_df):
        mode_key = "single" if mode_radio == "Single" else "batch"
        path = self.get_log_path(wm_name, mode_key)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        cols = ["Method", "Dataset Config", "Seed", "Attack Chain", "Metrics"]
        if new_df is None or new_df.empty:
            pd.DataFrame(columns=cols).to_csv(path, index=False)
        else:
            if len(new_df.columns) == len(cols): new_df.columns = cols
            new_df.to_csv(path, index=False)
        return None

    def modify_table(self, action, wm_name, mode_radio, current_df, selected_idx):
        if current_df is None or current_df.empty: return current_df
        if selected_idx is None: return current_df
        df = current_df.copy()
        idx = int(selected_idx)
        if action == "delete": df = df.drop(idx).reset_index(drop=True)
        elif action == "up":
            if idx > 0: df.iloc[idx], df.iloc[idx-1] = df.iloc[idx-1].copy(), df.iloc[idx].copy()
        elif action == "down":
            if idx < len(df) - 1: df.iloc[idx], df.iloc[idx+1] = df.iloc[idx+1].copy(), df.iloc[idx].copy()
        self.overwrite_logs(wm_name, mode_radio, df)
        return df

def log_format(tag, msg): return f">>> [{tag}] {msg}\n"

backend = BenchmarkBackend()




css_style = """
.gradio-container { max-width: 100% !important; width: 100% !important; margin: 0 !important; padding: 10px 20px !important; background-color: #f0f2f5; font-family: 'Inter', sans-serif; }
.header-container { text-align: center; background: linear-gradient(to right, #f8f9fa, #e9ecef); padding: 30px; border-radius: 0 0 15px 15px; color: #333; box-shadow: 0 4px 12px rgba(0,0,0,0.05); margin-bottom: 25px; border-bottom: 3px solid #3182ce; }
.header-title { font-size: 32px; font-weight: 800; letter-spacing: 1px; color: #1a202c; margin-bottom: 5px; }
.header-subtitle { font-size: 16px; color: #4a5568; font-weight: 400; margin-bottom: 15px; }
.status-bar { display: flex; gap: 25px; font-size: 13px; margin-top: 10px; justify-content: center; font-family: 'Courier New', monospace; color: #2d3748; }
.section-card { background: white; border-radius: 12px; padding: 24px; border: 1px solid #e5e7eb; box-shadow: 0 1px 3px 0 rgba(0, 0, 0, 0.1); margin-bottom: 20px; }
.section-header { font-size: 1.1rem; font-weight: 600; color: #111827; margin-bottom: 15px; display: flex; align-items: center; border-bottom: 2px solid #f3f4f6; padding-bottom: 10px; }
.section-header span { margin-right: 8px; font-size: 1.2rem; }
.btn-done { background: linear-gradient(45deg, #11998e, #38ef7d) !important; color: white !important; font-weight: bold !important; }
.btn-gray { background: #e5e7eb !important; color: #374151 !important; font-weight: 600 !important; }
.toggle-switch { display: flex; align-items: center; justify-content: space-between; padding: 0 5px; }
.toggle-switch label span { font-weight: 600; color: #374151; }
.toggle-switch input { appearance: none; width: 36px; height: 20px; background: #e5e7eb; border-radius: 999px; position: relative; cursor: pointer; transition: background-color 0.3s; flex-shrink: 0; }
.toggle-switch input::after { content: ''; position: absolute; top: 2px; left: 2px; width: 16px; height: 16px; background: white; border-radius: 50%; transition: transform 0.3s; }
.toggle-switch input:checked { background-color: #11998e; }
.toggle-switch input:checked::after { transform: translateX(16px); }
.footer { text-align: center; margin-top: 40px; padding: 20px; color: #6b7280; font-size: 0.875rem; border-top: 1px solid #e5e7eb; }
"""




with gr.Blocks(title="AI Watermark Benchmark", css=css_style, theme=gr.themes.Soft()) as app:
    
    
    gr.HTML("""
    <div class="header-container">
        <div class="header-title">🛡️ AI Watermark Benchmark</div>
        <div class="header-subtitle">Comprehensive Robustness Evaluation System for GenAI Watermarking</div>
        <div class="status-bar">
            <div class="status-item">System Status: <strong>Online</strong></div>
            <div class="status-item">Mode: <strong>Evaluation</strong></div>
        </div>
    </div>
    """)

    state_attacks = gr.State([False] * 8) 
    t1_seed_state = gr.State(2024) 
    state_atk_log = gr.State("Clean")
    state_current_img = gr.State(None)
    state_attacked_img = gr.State(None)
    state_selected_row = gr.State(None)

    
    default_exp = 'run_DwtDct.yaml' if 'run_DwtDct.yaml' in GLOBAL_LISTS['exp'] else (GLOBAL_LISTS['exp'][0] if GLOBAL_LISTS['exp'] else None)
    init_data = backend.load_experiment_template_data(default_exp)
    
    iv_wm = init_data['wm']
    iv_ds = init_data['ds']
    iv_seed = init_data['seed']
    iv_bs = init_data['bs']
    iv_max = init_data['max']
    iv_model = init_data['model']
    iv_states = init_data['states']
    iv_params = init_data['params']
    
    state_attacks_batch = gr.State(iv_states)

    with gr.Tabs():
        
        with gr.Tab("🧪 Single Evaluation"):
            with gr.Row():
                with gr.Column(scale=4, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'><span>⚙️</span> Generation Config</div>")
                    t1_wm_dd = gr.Dropdown(GLOBAL_LISTS['wm'], label="Watermark Algorithm", value="RivaGan.yaml")
                    t1_prompt = gr.Textbox(label="Prompt", value="A futuristic city, cyberpunk style, high detail", lines=2)
                    with gr.Row():
                        t1_seed = gr.Number(label="Seed", value=2024, precision=0)
                        t1_btn_gen = gr.Button("1. 🚀 Generate", elem_classes="btn-gray")
                    t1_status = gr.Textbox(label="System Status", interactive=False, max_lines=1)

                    gr.Markdown("<div class='section-header'><span>⚔️</span> Attack Suite</div>")
                    with gr.Accordion("Attack Settings", open=True):
                        with gr.Row():
                            t1_chk_0 = gr.Checkbox(label="Brightness", elem_classes="toggle-switch")
                            t1_chk_1 = gr.Checkbox(label="JPEG", elem_classes="toggle-switch")
                            t1_chk_2 = gr.Checkbox(label="Blur", elem_classes="toggle-switch")
                            t1_chk_3 = gr.Checkbox(label="Noise", elem_classes="toggle-switch")
                        with gr.Row():
                            t1_chk_4 = gr.Checkbox(label="Crop", elem_classes="toggle-switch")
                            t1_chk_5 = gr.Checkbox(label="Drop", elem_classes="toggle-switch")
                            t1_chk_6 = gr.Checkbox(label="VAE", elem_classes="toggle-switch")
                            t1_chk_7 = gr.Checkbox(label="Diff", elem_classes="toggle-switch")
                        gr.Markdown("**Parameters**")
                        sl_bright = gr.Slider(0.5, 2.0, value=1.2, label="Brightness Factor", step=0.1)
                        sl_jpeg = gr.Slider(10, 100, value=80, label="JPEG Quality", step=5)
                        sl_blur = gr.Slider(0.0, 5.0, value=1.0, label="Blur Radius", step=0.1)
                        sl_noise = gr.Slider(0.0, 0.2, value=0.05, label="Noise Std", step=0.01)
                        sl_crop = gr.Slider(0.5, 1.0, value=0.9, label="Crop Scale", step=0.05)
                        sl_drop = gr.Slider(0.0, 1.0, value=0.3, label="Drop Ratio", step=0.1)
                        with gr.Row():
                            sl_vae = gr.Slider(1, 8, value=3, step=1, label="VAE Quality")
                            t1_dd_vae_model = gr.Dropdown(["bmshj2018-factorized", "cheng2020-anchor"], value="bmshj2018-factorized", label="VAE Model", container=False, scale=1)
                        sl_diff = gr.Slider(0.0, 1.0, value=0.1, label="Diffusion Strength", step=0.05)
                    t1_btn_apply = gr.Button("2. ⚡ Apply Attacks", elem_classes="btn-gray")

                with gr.Column(scale=5):
                    with gr.Column(elem_classes="section-card"):
                        gr.Markdown("<div class='section-header'><span>👁️</span> Visual Comparison</div>")
                        with gr.Row():
                            t1_img_src = gr.Image(label="Original (Watermarked)", type="pil", interactive=False, height=300)
                            t1_img_dst = gr.Image(label="Attacked (To Verify)", type="pil", interactive=False, height=300)
                    with gr.Column(elem_classes="section-card"):
                        gr.Markdown("<div class='section-header'><span>📊</span> Detection Results</div>")
                        t1_btn_extract = gr.Button("3. 🔍 Extract & Verify", elem_classes="btn-gray")
                        with gr.Row():
                            t1_res_status = gr.Textbox(label="Verdict", show_label=False, text_align="center")
                            t1_res_text = gr.Code(label="Aggregated Metrics", language="yaml", lines=5)

        
        with gr.Tab("📊 Batch Benchmark"):
            with gr.Row():
                with gr.Column(scale=4, elem_classes="section-card"):
                    gr.Markdown("<div class='section-header'><span>📋</span> Batch Configuration</div>")
                    
                    t2_exp_dd = gr.Dropdown(GLOBAL_LISTS['exp'], label="Load Template", value=default_exp)
                    
                    with gr.Row():
                        t2_wm_dd = gr.Dropdown(GLOBAL_LISTS['wm'], label="Watermark", value=iv_wm)
                        t2_ds_dd = gr.Dropdown(GLOBAL_LISTS['ds'], label="Dataset", value=iv_ds)
                    
                    t2_model_dd = gr.Dropdown(GLOBAL_LISTS['model'], label="Model Config", value=iv_model)
                    
                    with gr.Row():
                        t2_seed = gr.Number(label="Seed", value=iv_seed, precision=0)
                        t2_bs = gr.Number(label="Batch Size", value=iv_bs, precision=0)
                        t2_max = gr.Number(label="Max Samples", value=iv_max, precision=0)
                    
                    gr.Markdown("<div class='section-header'><span>⚔️</span> Attack Chain</div>")
                    with gr.Accordion("Configure Batch Attacks", open=True):
                        with gr.Row():
                            t2_chk_0 = gr.Checkbox(label="Brightness", value=iv_states[0], elem_classes="toggle-switch")
                            t2_chk_1 = gr.Checkbox(label="JPEG", value=iv_states[1], elem_classes="toggle-switch")
                            t2_chk_2 = gr.Checkbox(label="Blur", value=iv_states[2], elem_classes="toggle-switch")
                            t2_chk_3 = gr.Checkbox(label="Noise", value=iv_states[3], elem_classes="toggle-switch")
                        with gr.Row():
                            t2_chk_4 = gr.Checkbox(label="Crop", value=iv_states[4], elem_classes="toggle-switch")
                            t2_chk_5 = gr.Checkbox(label="Drop", value=iv_states[5], elem_classes="toggle-switch")
                            t2_chk_6 = gr.Checkbox(label="VAE", value=iv_states[6], elem_classes="toggle-switch")
                            t2_chk_7 = gr.Checkbox(label="Diff", value=iv_states[7], elem_classes="toggle-switch")
                        
                        gr.Markdown("**Batch Parameters**")
                        t2_sl_bright = gr.Slider(0.5, 2.0, value=iv_params[0], label="Brightness", step=0.1)
                        t2_sl_jpeg = gr.Slider(10, 100, value=iv_params[1], label="JPEG", step=5)
                        t2_sl_blur = gr.Slider(0.0, 5.0, value=iv_params[2], label="Blur", step=0.1)
                        t2_sl_noise = gr.Slider(0.0, 0.2, value=iv_params[3], label="Noise", step=0.01)
                        t2_sl_crop = gr.Slider(0.5, 1.0, value=iv_params[4], label="Crop", step=0.05)
                        t2_sl_drop = gr.Slider(0.0, 1.0, value=iv_params[5], label="Drop", step=0.1)
                        with gr.Row():
                            t2_sl_vae = gr.Slider(1, 8, value=iv_params[6], step=1, label="VAE Quality")
                            t2_dd_vae_model = gr.Dropdown(["bmshj2018-factorized", "cheng2020-anchor"], value="bmshj2018-factorized", label="VAE Model", container=False, scale=1)
                        t2_sl_diff = gr.Slider(0.0, 1.0, value=iv_params[7], label="Diff Strength", step=0.05)
                    
                    t2_btn_run = gr.Button("🚀 Run Batch Benchmark", elem_classes="btn-done")

                with gr.Column(scale=5):
                    with gr.Column(elem_classes="section-card"):
                        gr.Markdown("<div class='section-header'><span>💻</span> Execution Logs</div>")
                        t2_log_out = gr.TextArea(label="Terminal Output", lines=20, interactive=False, autoscroll=True)
                        t2_res_text = gr.Code(label="Final Results Summary", language="yaml", lines=5)

    with gr.Column(elem_classes="section-card"):
        gr.Markdown("<div class='section-header'><span>📝</span> Experiment History</div>")
        with gr.Row():
            hist_wm_dd = gr.Dropdown(GLOBAL_LISTS['wm'], label="Filter by Watermark", value="RivaGan.yaml")
            hist_mode_radio = gr.Radio(["Single", "Batch"], label="View Mode", value="Single")
            btn_refresh = gr.Button("🔄 Refresh Table", elem_classes="btn-secondary")
        with gr.Row():
            btn_row_up = gr.Button("⬆️ Move Up", size="sm")
            btn_row_down = gr.Button("⬇️ Move Down", size="sm")
            btn_row_del = gr.Button("🗑️ Delete Selected Row", variant="stop", size="sm")
        log_table = gr.Dataframe(value=backend.load_logs("RivaGan.yaml", "Single"), headers=["Method", "Dataset Config", "Seed", "Attack Chain", "Metrics"], interactive=False, col_count=(5, "fixed"), datatype=["str", "str", "number", "str", "str"])

    gr.HTML("""<div class="footer"><p>© 2024 AI Watermark Benchmark System | Powered by Gradio</p></div>""")

    
    
    t1_btn_gen.click(backend.generate_and_embed, inputs=[t1_prompt, t1_seed, t1_wm_dd], outputs=[t1_img_src, t1_img_dst, state_current_img, state_attacked_img, t1_status, t1_seed_state, t1_btn_gen])
    
    def run_attack_wrapper_t1(hidden_img, c0,c1,c2,c3,c4,c5,c6,c7, b, j, bl, n, c, d, v, df, vae_m):
        states = [c0,c1,c2,c3,c4,c5,c6,c7]
        params = {'bright': b, 'jpeg': j, 'blur': bl, 'noise': n, 'crop': c, 'drop': d, 'vae_qual': v, 'diff_str': df, 'vae_model': vae_m}
        return backend.apply_attacks(hidden_img, states, params)
    
    t1_btn_apply.click(run_attack_wrapper_t1, inputs=[state_current_img, t1_chk_0, t1_chk_1, t1_chk_2, t1_chk_3, t1_chk_4, t1_chk_5, t1_chk_6, t1_chk_7, sl_bright, sl_jpeg, sl_blur, sl_noise, sl_crop, sl_drop, sl_vae, sl_diff, t1_dd_vae_model], outputs=[t1_img_dst, state_attacked_img, state_atk_log, t1_btn_apply])
    
    def extract_wrapper_t1(hidden_img, wm, atk_str, seed_val, current_df, view_wm, view_mode):
        met, stat, new_btn = backend.extract_and_verify(hidden_img)
        metrics_inline = met.replace('\n', ', ')
        backend.save_log_entry(wm, "Single_Image", seed_val, atk_str, metrics_inline, mode="single")
        if wm == view_wm and view_mode == "Single":
            return met, stat, backend.load_logs(wm, "Single"), new_btn
        return met, stat, current_df, new_btn
    
    t1_btn_extract.click(extract_wrapper_t1, inputs=[state_attacked_img, t1_wm_dd, state_atk_log, t1_seed_state, log_table, hist_wm_dd, hist_mode_radio], outputs=[t1_res_text, t1_res_status, log_table, t1_btn_extract])

    
    
    
    t2_ui_outputs = [
        t2_exp_dd, 
        t2_wm_dd, 
        t2_ds_dd, 
        t2_model_dd,
        t2_seed, t2_bs, t2_max,
        t2_chk_0, t2_chk_1, t2_chk_2, t2_chk_3, t2_chk_4, t2_chk_5, t2_chk_6, t2_chk_7,
        t2_sl_bright, t2_sl_jpeg, t2_sl_blur, t2_sl_noise, t2_sl_crop, t2_sl_drop, t2_sl_vae, t2_sl_diff
    ]

    
    t2_exp_dd.change(
        backend.load_experiment_template_ui, 
        inputs=[t2_exp_dd], 
        outputs=t2_ui_outputs[1:] 
    )

    
    t2_wm_dd.change(
        backend.find_template_by_watermark,
        inputs=[t2_wm_dd],
        outputs=t2_ui_outputs
    )

    def run_batch_bridge(wm, ds, model, seed, bs, b_max, c0,c1,c2,c3,c4,c5,c6,c7, b, j, bl, n, c, d, v, df, vae_m):
        states = [c0,c1,c2,c3,c4,c5,c6,c7]
        params = {'bright': b, 'jpeg': j, 'blur': bl, 'noise': n, 'crop': c, 'drop': d, 'vae': v, 'diff': df, 'vae_model': vae_m}
        for log, res in backend.run_batch_benchmark(wm, ds, model, seed, bs, b_max, states, params): yield log, res

    t2_btn_run.click(
        run_batch_bridge, 
        inputs=[
            t2_wm_dd, t2_ds_dd, t2_model_dd, t2_seed, t2_bs, t2_max, 
            t2_chk_0, t2_chk_1, t2_chk_2, t2_chk_3, t2_chk_4, t2_chk_5, t2_chk_6, t2_chk_7,
            t2_sl_bright, t2_sl_jpeg, t2_sl_blur, t2_sl_noise, t2_sl_crop, t2_sl_drop, t2_sl_vae, t2_sl_diff, t2_dd_vae_model
        ], 
        outputs=[t2_log_out, t2_res_text]
    )

    
    hist_wm_dd.change(backend.load_logs, inputs=[hist_wm_dd, hist_mode_radio], outputs=[log_table])
    hist_mode_radio.change(backend.load_logs, inputs=[hist_wm_dd, hist_mode_radio], outputs=[log_table])
    btn_refresh.click(backend.load_logs, inputs=[hist_wm_dd, hist_mode_radio], outputs=[log_table])
    log_table.change(backend.overwrite_logs, inputs=[hist_wm_dd, hist_mode_radio, log_table], outputs=[])
    def on_select(evt: gr.SelectData): return evt.index[0]
    log_table.select(on_select, None, state_selected_row)
    btn_row_del.click(lambda wm, m, df, idx: backend.modify_table("delete", wm, m, df, idx), inputs=[hist_wm_dd, hist_mode_radio, log_table, state_selected_row], outputs=[log_table])
    btn_row_up.click(lambda wm, m, df, idx: backend.modify_table("up", wm, m, df, idx), inputs=[hist_wm_dd, hist_mode_radio, log_table, state_selected_row], outputs=[log_table])
    btn_row_down.click(lambda wm, m, df, idx: backend.modify_table("down", wm, m, df, idx), inputs=[hist_wm_dd, hist_mode_radio, log_table, state_selected_row], outputs=[log_table])
    app.load(backend.load_logs, inputs=[hist_wm_dd, hist_mode_radio], outputs=[log_table])

if __name__ == "__main__":
    app.queue().launch(share=True)